# Groq Multilingual Chatbot — Colab Test

This notebook tests a simple voice → text → multilingual chatbot pipeline using separate Groq API keys and models for each component.

Pipeline:

`Audio → Whisper Large v3 → GPT OSS 20B → Text response`

We keep STT and LLM clients separate so either component can be swapped independently.

## 1. Install dependencies


In [1]:
!pip -q install -U groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.2 MB/s eta 0:00:00


## 2. Configure Colab Secrets

In Colab, open **Secrets** (key icon) and add:

- `GROQ_API_KEY_STT` — key used only for speech-to-text
- `GROQ_API_KEY_LLM` — key used only for the chatbot

Do **not** paste API keys directly into notebook cells.

In [2]:
from google.colab import userdata

GROQ_API_KEY_STT = userdata.get("GROQ_API_KEY_STT")
GROQ_API_KEY_LLM = userdata.get("GROQ_API_KEY_LLM")

if not GROQ_API_KEY_STT:
    raise ValueError("Missing Colab Secret: GROQ_API_KEY_STT")
if not GROQ_API_KEY_LLM:
    raise ValueError("Missing Colab Secret: GROQ_API_KEY_LLM")

print("STT key loaded:", bool(GROQ_API_KEY_STT))
print("LLM key loaded:", bool(GROQ_API_KEY_LLM))

STT key loaded: True
LLM key loaded: True


## 3. Choose models

`whisper-large-v3` is used for multilingual speech recognition.

`openai/gpt-oss-20b` is used for multilingual dialogue. We use instruct/non-thinking mode for normal chatbot responses.

In [3]:
STT_MODEL = "whisper-large-v3"
LLM_MODEL = "openai/gpt-oss-20b"

print("STT:", STT_MODEL)
print("LLM:", LLM_MODEL)

STT: whisper-large-v3
LLM: openai/gpt-oss-20b


## 4. Create separate Groq clients


In [4]:
from groq import Groq

stt_client = Groq(api_key=GROQ_API_KEY_STT)
llm_client = Groq(api_key=GROQ_API_KEY_LLM)

print("Groq clients ready.")

Groq clients ready.


## 5. Test the LLM independently

This lets us verify the chatbot model before dealing with audio.

In [5]:
import time

test_prompt = "मराठीत माझ्याशी मैत्रीपूर्ण पद्धतीने बोला. कृपया स्वतःची थोडक्यात ओळख करून द्या."

start = time.perf_counter()

completion = llm_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a helpful multilingual chatbot. Reply in the same language as the user unless asked otherwise. Keep normal conversational replies concise."
        },
        {"role": "user", "content": test_prompt}
    ],
    reasoning_effort="medium",
    temperature=0.7,
    max_completion_tokens=300,
)

elapsed = time.perf_counter() - start
llm_text = completion.choices[0].message.content

print("Response:\n")
print(llm_text)
print(f"\nLatency: {elapsed:.2f}s")

Response:

नमस्कार! मी ChatGPT, तुमचा मित्रवत आभासी सहाय्यक. मला तुमचे प्रश्न उत्तर द्यायला, माहिती शोधायला, आणि संवाद साधायला आवडते. काही विचारायचे असेल तर निःसंकोच विचारा!

Latency: 0.43s


## 6. Upload an audio file

Upload a short `.wav`, `.mp3`, `.m4a`, `.ogg`, or `.webm` recording. For the first test, keep it short.

In [6]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode

def record_audio(seconds=8, filename="recording.webm"):
    display(Javascript("""
    async function recordAudio(seconds) {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];

      recorder.ondataavailable = event => chunks.push(event.data);
      recorder.start();

      await new Promise(resolve => setTimeout(resolve, seconds * 1000));

      recorder.stop();
      await new Promise(resolve => recorder.onstop = resolve);

      stream.getTracks().forEach(track => track.stop());

      const blob = new Blob(chunks, {type: 'audio/webm'});
      const reader = new FileReader();

      return await new Promise(resolve => {
        reader.onloadend = () => resolve(reader.result);
        reader.readAsDataURL(blob);
      });
    }
    """))

    data = eval_js(f"recordAudio({seconds})")
    audio = b64decode(data.split(",")[1])

    with open(filename, "wb") as f:
        f.write(audio)

    return filename

In [49]:
audio_path = record_audio(seconds=3)

print("Recording complete!")

<IPython.core.display.Javascript object>

Recording complete!


## 7. Transcribe with Whisper Large v3

Leave `LANGUAGE = None` for automatic language detection. If you know the language, setting an ISO-639-1 code such as `mr`, `hi`, or `en` can improve accuracy and latency.

In [50]:
LANGUAGE = None  # "mr", "hi", "en", "ta", etc. or None for automatic detection

with open(audio_path, "rb") as audio_file:
    start = time.perf_counter()

    transcription = stt_client.audio.transcriptions.create(
        file=(audio_path, audio_file.read()),
        model=STT_MODEL,
        language=LANGUAGE,
        response_format="json",
        temperature=0.0,
    )

    stt_elapsed = time.perf_counter() - start

transcript = transcription.text.strip()

print("You said:")
print(transcript)
print(f"\nSTT latency: {stt_elapsed:.2f}s")

You said:
तमे केम छो

STT latency: 0.27s


## 8. Send the transcription to the multilingual LLM


In [51]:
start = time.perf_counter()

response = llm_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a helpful multilingual voice chatbot. Answer naturally in the same language as the user's message. Do not translate the user's message unless asked. Keep answers concise for voice output."
        },
        {"role": "user", "content": transcript}
    ],
    reasoning_effort="medium",
    temperature=0.7,
    max_completion_tokens=500,
)

llm_elapsed = time.perf_counter() - start
answer = response.choices[0].message.content.strip()

print("Bot response:\n")
print(answer)
print(f"\nLLM latency: {llm_elapsed:.2f}s")

Bot response:

હું સારું છું, આભાર. તમે કેવી છો?

LLM latency: 0.17s


In [17]:
!pip -q install gTTS

In [53]:
from gtts import gTTS
from IPython.display import Audio, display

answer = response.choices[0].message.content.strip()

TTS_LANGUAGE = "gu"  # Marathi
# "hi" = Hindi
# "en" = English
# "ta" = Tamil
# "te" = Telugu

tts = gTTS(text=answer, lang=TTS_LANGUAGE)
tts.save("response.mp3")

display(Audio("response.mp3", autoplay=True))

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display

answer = response.choices[0].message.content.strip()

TTS_LANGUAGE = "te"  # Marathi
# "hi" = Hindi
# "en" = English
# "ta" = Tamil
# "te" = Telugu

tts = gTTS(text=answer, lang=TTS_LANGUAGE)
tts.save("response.mp3")

display(Audio("response.mp3", autoplay=True))

## 9. Full pipeline test

This function makes it easy to repeatedly test different languages and recordings.

In [ ]:
def voice_chat(audio_path, language=None):
    total_start = time.perf_counter()

    # STT
    with open(audio_path, "rb") as audio_file:
        stt_start = time.perf_counter()
        transcription = stt_client.audio.transcriptions.create(
            file=(audio_path, audio_file.read()),
            model=STT_MODEL,
            language=language,
            response_format="json",
            temperature=0.0,
        )
        stt_time = time.perf_counter() - stt_start

    text = transcription.text.strip()

    # LLM
    llm_start = time.perf_counter()
    completion = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful multilingual voice chatbot. Reply naturally in the same language as the user. Keep responses concise and suitable for spoken output."
            },
            {"role": "user", "content": text},
        ],
        reasoning_effort="medium",
        temperature=0.7,
        max_completion_tokens=500,
    )
    llm_time = time.perf_counter() - llm_start

    answer = completion.choices[0].message.content.strip()
    total_time = time.perf_counter() - total_start

    return {
        "transcript": text,
        "answer": answer,
        "stt_seconds": round(stt_time, 3),
        "llm_seconds": round(llm_time, 3),
        "total_seconds": round(total_time, 3),
    }

result = voice_chat(audio_path, language=LANGUAGE)
result

{'transcript': 'میرا نام ارنب ہے آپ کا نام کیا ہے؟',
 'answer': 'میرا نام ChatGPT ہے۔',
 'stt_seconds': 0.534,
 'llm_seconds': 0.34,
 'total_seconds': 0.875}

## 10. What to test

Try separate recordings in:

- English
- Hindi (`hi`)
- Marathi (`mr`)
- Tamil (`ta`)
- Telugu (`te`)

Record the same type of sentence in each language and compare transcription quality, response quality, and latency.

### Next stage

After this works, add:

1. microphone recording directly inside Colab
2. conversation history / memory
3. language detection
4. streaming responses
5. text-to-speech
6. a simple web UI

Keep STT and LLM API keys separate as the project grows.